In [1]:
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import torch
import os
from jobable.ml_logic.model import tokenizer, model

/Users/jonny/.pyenv/versions/jobable/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model from: /Users/jonny/code/JonnyBeAverage/Jobable/jobable/model_weights/cover_letter_model
Using device: mps


Loading weights: 100%|██████████| 453/453 [00:00<00:00, 1535.38it/s, Materializing param=model.layers.31.self_attn.v_proj.weight]


In [2]:
dataset = load_dataset("ShashiVish/cover-letter-dataset")

In [3]:
dataset['train'].features

{'Job Title': Value('string'),
 'Preferred Qualifications': Value('string'),
 'Hiring Company': Value('string'),
 'Applicant Name': Value('string'),
 'Past Working Experience': Value('string'),
 'Current Working Experience': Value('string'),
 'Skillsets': Value('string'),
 'Qualifications': Value('string'),
 'Cover Letter': Value('string')}

In [4]:
def preprocess(row):
    X = f'''
    Job title: {row['Job Title']} ;
    Qualifications: {row['Preferred Qualifications']} ;
    Hiring Company: {row['Hiring Company']} ;
    Applicant Name: {row['Applicant Name']} ;
    Past Working Experience: {row['Past Working Experience']} ;
    Current Working Experience: {row['Current Working Experience']} ;
    Skillsets: {row['Skillsets']} ;
    Qualifications: {row['Qualifications']} ;
    '''

    y = row['Cover Letter']

    return {'input_text': X, 'target_text': y}

dataset = dataset.map(preprocess)

In [5]:
def tokenize(row):
    model_inputs = tokenizer(
        row['input_text'],
        max_length=512,
        truncation=True,
        padding='max_length'
        )

    labels = tokenizer(
        row['target_text'],
        max_length=512,
        truncation=True,
        padding='max_length'
        )

    model_inputs['labels'] = labels['input_ids']

    return model_inputs

dataset = dataset.map(tokenize, batched=True)

Map: 100%|██████████| 349/349 [00:00<00:00, 3273.86 examples/s]


In [6]:
os.environ["WANDB_DISABLED"] = "true"

In [15]:
# os.makedirs("./cover_letter_model", exist_ok=True)
training_args = TrainingArguments(
    output_dir="./cover_letter_model",
    eval_strategy="epoch",  # Updated deprecated argument
    save_strategy="epoch",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    fp16=True,
    weight_decay=0.01,
    save_total_limit=2,
    report_to=["none"]  # Disabling Weights & Biases properly
)

In [16]:
model.gradient_checkpointing_enable()

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)


In [18]:
torch.mps.empty_cache()

In [19]:
trainer.train()


RuntimeError: MPS backend out of memory (MPS allocated: 18.09 GiB, other allocations: 34.72 MiB, max allowed: 18.13 GiB). Tried to allocate 8.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
model.save_pretrained("./cover_letter_model")
tokenizer.save_pretrained("./cover_letter_model")
